# RoofTop Regression Scoping

## Purpose

Scope the APTs (Address Points) that were **relocated for ESP** so they can be validated
and, once relocation is confirmed, fed into the Data Correctness (relocation) process.

### Background (SEACO-6171)

In Q1–Q2 2026, a total of **8,569,598 (~8.5M)** APTs were ingested across multiple sources
(e.g. `CATASTRO_APT`), with locations updated according to each respective source. Some of
these ingestions moved existing APTs (originally from Genesis, Nomecalles Opensource,
APT Eustat, esp event_lost) away from their rooftop, causing a regression in the ESP RAC
(Rooftop Accuracy) metrics.

### Scope of this notebook

As the next step in the investigation, this notebook:

1. Determines the exact count of APTs that moved **outside the Building Footprint (BFP)**
   in Q1 and Q2 for ESP.
2. Scopes those relocated APTs.
3. Prepares the scoped set for validation, after which the Data Correctness process for
   relocation can be triggered.

Reference: [SEACO-6171](https://tomtom.atlassian.net/browse/SEACO-6171?focusedCommentId=12144061)

## Approach

The relocated-APT scoping is built step by step from two snapshots (OLD vs LATEST) and the
Building Footprint (BFP) layer:

1. **Load the OLD snapshot** into a Delta table (pin the older revision to compare against).
2. **Load the LATEST snapshot** into a Delta table (resolve the newest revision).
3. **Read the OLD snapshot** Delta table as a `Dataset[OrbisElement]`.
4. **Read the LATEST snapshot** Delta table as a `Dataset[OrbisElement]`.
5. **Filter both snapshots by `countryISO` = `ESP`** (value of the `metadata:country` tag).
6. **Filter the OLD snapshot by `metadata:updated`** — keep only APTs whose `metadata:updated`
   value (an epoch timestamp) is **greater than 1st Feb** (the regression window start).
7. **Left join** the OLD snapshot dataset with the LATEST snapshot dataset on the APT id
   (keep all OLD rows; LATEST columns are null where the APT is missing in the latest).
8. **Add geometry / distance columns:**
   - `old_wkt` and `latest_wkt` — `POINT(lng lat)` built from each snapshot's lat/lng.
   - `distance_in_meters` — great-circle (Haversine) distance between the OLD and LATEST point.
   - `hookline` — `LINESTRING(old -> latest)` connecting the two points.
9. **Load the Building Footprint (BFP) snapshot.**
10. **Spatially check the OLD location** of each APT against the BFP — was the OLD APT
    point inside a building footprint?
11. **Final dataset** contains, per APT:
    - OLD snapshot details (location, tags),
    - LATEST snapshot details (location, tags),
    - `distance_in_meters` and `hookline`,
    - whether the **OLD** APT was inside a BFP, and whether the **NEW** APT is inside a BFP.
